# 03 — The Kantorovich LP

Relax Monge's map into a *coupling* that may split mass, and optimal transport becomes a linear program that always has a solution.

**Prerequisites:** `02_where_monge_breaks`.

**What you'll be able to do**
- Write a transport plan as a coupling $P$ with row- and column-marginal constraints.
- Solve the Kantorovich linear program and read the optimal coupling, splitting and all.
- Recognise the transportation polytope $T(a, b)$ and see why it is never empty.

Last notebook, a target sat stubbornly empty because Monge's map could not split mass. Kantorovich's relaxation changes a single word — *map* becomes *coupling* — and everything loosens: the empty target fills, the problem turns into a linear program, and a solution always exists. This LP is the workhorse of the whole module; we will measure it (Wasserstein), dualise it (Sinkhorn), and quantise it (Module 4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.transport.discrete import (
    squared_euclidean_cost,
    discrete_ot_plan,
    transport_cost,
    is_transportation_polytope_member,
)

np.random.seed(42)
viz.use_course_style()

## 1. From a map to a coupling

Monge failed because a function cannot split mass. Kantorovich's fix is to describe transport not by a map but by a **coupling**: a matrix $P \in \mathbb{R}_+^{n \times m}$ whose entry $P_{ij}$ is the amount of mass carried from source $i$ to target $j$. A single source row may now hold several non-zero entries — its mass *fans out* across targets.

A coupling has to respect both piles, so its row sums return the source masses and its column sums return the target masses:

$$ P\,\mathbf{1} = a, \qquad P^\top \mathbf{1} = b, \qquad P \ge 0. $$

Among all such couplings, Kantorovich seeks the cheapest under the ground cost — a **linear program**:

$$ \min_{P \ge 0}\ \langle C, P\rangle = \sum_{i,j} P_{ij}\, C_{ij} \quad\text{subject to}\quad P\mathbf{1}=a,\ P^\top\mathbf{1}=b. $$

In [ ]:
# The very example that defeated Monge in the previous notebook.
x_source = np.array([0.0, 4.0])
x_target = np.array([1.0, 2.0, 3.0])
a = np.array([0.5, 0.5])
b = np.array([1 / 3, 1 / 3, 1 / 3])

C = squared_euclidean_cost(x_source, x_target)   # C[i, j] = (x_source[i] - x_target[j]) ** 2
plan = discrete_ot_plan(a, b, C)                  # solve the Kantorovich LP (POT network simplex)

print("optimal coupling P =")
print(np.round(plan, 4))
print(f"\nrow sums {np.round(plan.sum(axis=1), 3).tolist()}  ==  a {a.tolist()}")
print(f"col sums {np.round(plan.sum(axis=0), 3).tolist()}  ==  b {np.round(b, 3).tolist()}")
print(f"valid coupling in T(a, b)?  {is_transportation_polytope_member(plan, a, b)}")
print(f"non-zero entries: {int(np.count_nonzero(plan > 1e-9))}  (a vertex has at most n + m - 1 = {len(a) + len(b) - 1})")
print(f"\ntransport cost <C, P> = {transport_cost(plan, C):.4f}")
print(f"cross-check ot.emd2    = {ot.emd2(a, b, C):.4f}")

**Read the output.** Both source rows now carry mass to more than one target — the splitting that no Monge map could express. The row sums return $a$ and the column sums return $b$, so the push-forward that was impossible for a function holds exactly for this coupling, and the once-empty middle target is filled. The solution sits at a vertex of the feasible set, so it uses at most $n + m - 1$ non-zero routes. Kantorovich solved the very problem that broke Monge — by allowing the split.

## 2. The transportation polytope is never empty

The feasible set — all couplings with the right marginals — is the **transportation polytope**

$$ T(a, b) = \{ P \ge 0 :\ P\mathbf{1} = a,\ P^\top\mathbf{1} = b \}. $$

It is a bounded convex set carved out by linear constraints, and it is **never empty**: the *independent coupling* $P^{\mathrm{indep}} = a\,b^\top$ — ignore positions, match mass proportionally — always meets the marginals. So Kantorovich's LP always has at least one feasible point to optimise over, even where Monge has none. The LP picks the cheapest member of $T(a, b)$; the independent coupling is merely the most blind one.

In [ ]:
P_indep = a[:, None] * b[None, :]              # the independent coupling  a b^T
print("independent coupling  a b^T =")
print(np.round(P_indep, 4))
print(f"feasible (in T(a, b))?  {is_transportation_polytope_member(P_indep, a, b)}")
print(f"its cost  <C, P_indep> = {transport_cost(P_indep, C):.4f}")
print(f"optimal cost <C, P>    = {transport_cost(plan, C):.4f}")

**Read the output.** The independent coupling is a perfectly valid member of $T(a, b)$ — it satisfies every marginal — but it costs markedly more than the optimum, because it spreads mass without regard to distance. That gap between *blind* feasibility and *optimal* transport is the whole value of solving the LP. (Keep this coupling in mind: in the Sinkhorn arc it returns as the limit of entropic transport when regularisation is turned all the way up.)

In [ ]:
viz.plot_cost_matrix(C)
plt.show()

viz.plot_transport_plan(plan, title="Optimal coupling P  (rows = source, cols = target)")
plt.show()

viz.plot_transport_arrows(
    a, b, plan,
    source_positions=x_source, target_positions=x_target,
    title="Kantorovich solves it: source mass splits to fill every target",
)
plt.show()

**Read the three figures.**

- **Cost matrix.** The cheap routes (dark) hug the near-diagonal pairings; carrying mass across the full span (source at $0$ to target at $3$) is the brightest, most expensive cell.
- **Plan heatmap.** Each source row lights up in more than one column — that visible spread *is* the mass splitting, the thing a single-arrow map could never draw.
- **Flow view.** Every source pile now fans out to several targets, and crucially a line reaches the middle target that stood empty last time. The push-forward holds, and it holds cheaply.

## Your turn

1. **Watch the plan adapt.** Move the right source from $4$ to $2.5$ and re-solve. Where does the mass go now, and does any row stop splitting?
2. **Cost of blindness.** Compute the ratio `transport_cost(P_indep, C) / transport_cost(plan, C)` for this example, then for a target bunched near the sources. When is the independent coupling almost as good as optimal — and why?
3. **Read a vertex.** A basic feasible solution has at most $n + m - 1$ non-zero entries. Count the non-zeros in the optimal `plan` and check the bound. Can you construct marginals where the optimum uses strictly fewer?

## What you built

- You wrote a transport plan as a coupling $P$ obeying $P\mathbf{1} = a$, $P^\top\mathbf{1} = b$, $P \ge 0$.
- You solved the Kantorovich LP and read the optimal coupling — both rows splitting to fill every target.
- You met the transportation polytope $T(a, b)$ and saw it is never empty, so the LP always has a solution.

You now hold the central object of classical optimal transport. Every distance, dual, and algorithm in the rest of the module is built on this one linear program — and you can already solve it and read its answer.

## What's next

In `04_birkhoff_and_assignment` we specialise to uniform, equal-size marginals. The polytope becomes the **Birkhoff polytope** of doubly stochastic matrices, its corners turn out to be permutations, and discrete OT collapses to the classical **assignment problem** — the synthesis that closes this arc and opens the classical↔quantum dictionary.

## References

- L. V. Kantorovich, "On the translocation of masses", *Doklady Akademii Nauk SSSR* 37, 199–201 (1942).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 2–3. DOI:10.1561/2200000073
- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 1.

**Previous:** `notebooks/03_ClassicalOptimalTransport/02_where_monge_breaks.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/04_birkhoff_and_assignment.ipynb`